# 42 — BoneMarrowWSI: per-session (per-reader) cell labels for the verifier-error analysis (P1-4, RQ4)

**Why.** Every certificate in the paper assumes that verification is perfect: the expert's label is the truth. BoneMarrowWSI
publishes, besides the consensus, the label of each annotation *session*: session 0 is the first reader's label for every
cell, session 1 the second reader's, and sessions 2-5 the further readers who were asked only when earlier readers
disagreed (dataset documentation, Zenodo 10.5281/zenodo.14933087; which person read which cell is not recorded). Comparing a
single reader with the consensus tells us how a single verifier's errors would affect certification.

**What it does.** Downloads only the small annotation series (about 15 MB in total; no slide images), parses every labelled
box with the same code as `scripts/22_parse_bmwsi.py`, and saves one compressed table to
`MyDrive/certified-adaptive-pbs/bmwsi_sessions_v1/`: `annotations_sessions.csv.gz` (about 3 MB), `series_index.csv`,
`config.json`.

**Runtime:** CPU is enough (no GPU). Runtime > Run all; about 2-5 minutes. Written 2026-10-20; not yet run.


In [ ]:
# 0. Setup
!pip -q install idc-index highdicom==0.28.2 2>/dev/null
import os, glob, json, time, hashlib, platform
import numpy as np, pandas as pd, pydicom, highdicom as hd
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/certified-adaptive-pbs/bmwsi_sessions_v1'; os.makedirs(OUT, exist_ok=True)
from idc_index import IDCClient
c = IDCClient(); d = c.index
d = d[(d.collection_id == 'bonemarrowwsi_pediatricleukemia') & (d.Modality == 'ANN')]
keep = d[d.SeriesDescription.str.startswith('Session') | d.SeriesDescription.str.startswith('Consensus')]
print(keep.SeriesDescription.value_counts().sort_index(), round(keep.series_size_MB.sum(), 1), 'MB')
c.download_from_selection(seriesInstanceUID=list(keep.SeriesInstanceUID), downloadDir='/content/bm_sess')
keep[['PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SeriesDescription', 'series_size_MB']].to_csv(f'{OUT}/series_index.csv', index=False)


In [ ]:
# 1. Parse every labelled box (same logic as scripts/22_parse_bmwsi.py); source = 'consensus' or 'session<k>'
rows = []; t0 = time.time()
for f in sorted(glob.glob('/content/bm_sess/bonemarrowwsi_pediatricleukemia/**/*.dcm', recursive=True)):
    ds = pydicom.dcmread(f); desc = ds.SeriesDescription; pid = ds.PatientID
    src = 'consensus' if desc.startswith('Consensus') else 'session' + desc.split(':')[0].split()[-1]
    ann = hd.ann.MicroscopyBulkSimpleAnnotations.from_dataset(ds)
    for g in ann.get_annotation_groups():
        for cc in g.get_graphic_data(coordinate_type=ds.AnnotationCoordinateType):
            cc = np.asarray(cc, float)
            rows.append((pid, src, g.label, cc[:, 0].min(), cc[:, 1].min(), cc[:, 0].max(), cc[:, 1].max()))
A = pd.DataFrame(rows, columns=['patient', 'source', 'label', 'x0', 'y0', 'x1', 'y1'])
print(len(A), 'labelled boxes |', f'{time.time()-t0:.0f}s'); print(A.groupby('source').agg(boxes=('label', 'size'), patients=('patient', 'nunique')))
# check: the boxes of session 0 should be the consensus boxes (same coordinates)
key = lambda D: set(map(tuple, D[['patient', 'x0', 'y0', 'x1', 'y1']].round(0).astype(str).values))
kc, k0 = key(A[A.source == 'consensus']), key(A[A.source == 'session0'])
print('consensus boxes also in session 0:', round(len(kc & k0) / max(len(kc), 1), 4))


In [ ]:
# 2. Save
p = f'{OUT}/annotations_sessions.csv.gz'; A.to_csv(p, index=False, compression='gzip')
h = hashlib.sha256(open(p, 'rb').read()).hexdigest()
json.dump(dict(notebook='42_bmwsi_sessions_colab.ipynb', created_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
               n_boxes=len(A), by_source=A.source.value_counts().to_dict(), sha256=h, highdicom=hd.__version__,
               pydicom=pydicom.__version__, python=platform.python_version()), open(f'{OUT}/config.json', 'w'), indent=1)
print('saved', p, round(os.path.getsize(p) / 1e6, 2), 'MB  sha256', h[:16])


## After it finishes
Copy `bmwsi_sessions_v1/` to `data/bmwsi_sessions/` in the repo; then run `scripts/43_verifier_error.py`.